# Session 3:   Accessing S3 data
## *Prof. Gary Pavlis*

The learning objectives for this exercise are:

- You will learn how to set up a client in GeoLab to connect with the waveform archives
- You will learn how to do the equivalent of unix "cd" and "ls" commands to "list_objects_v2" and python coding
- You will learn the basic structure of the waveform archives in Earthscope's "S3 bucket".
- You will learn how to retrieve and manipulate waveform data stored in S3
- You will learn how to set up a worker plugin needed to utilize dask

A key reference for this material is the (evolving) Earthscope documentation on the [Earthscope SDK](https://docs.earthscope.org/sdk).  There are also an overwhelming number of sources on cloud computing in general and Amazon's boto3 package in particular (boto3 is the python module we will be using).   Any of the current AIs tend to give you pretty good information on all things S3 so consider that a prime source for learning about this material.

## Creating the client to access the archives
The package Amazon support for access to their cloud system is called "boto3".   Google tells me the name is more-or-less some programmers random association with a pink dolphin in the Amazon.   "3" just means it is version 3.  How is that for a trivial fact for the day? 

Earthscope built their python SDK (Software Development Kit) on top of boto3. There are some implementation details they describe in their documentation you can read at your leisure on the web site I gave you above.   For getting started you can just plan to copy the following code box for any serial processing on GeoLab.   One more example of the phrase you have heard me use many times by now - an incantation.  

Run this box and we'll poke around what it produces a bit below.

In [1]:
import boto3
from botocore.config import Config
from earthscope_sdk import EarthScopeClient

client = EarthScopeClient()
creds = client.user.get_aws_credentials()

S3_ACCESS_POINT = "earthscope-mseed-res-na3mtd4fq5kz7pntcyr1uh46use2a--ol-s3"
BUCKET = S3_ACCESS_POINT

session = boto3.Session(
    aws_access_key_id=creds.aws_access_key_id,
    aws_secret_access_key=creds.aws_secret_access_key.get_secret_value(),
    aws_session_token=creds.aws_session_token.get_secret_value(),
)
s3_client = session.client(
                    "s3",
                    config=Config(
                        request_checksum_calculation="when_required",
                        response_checksum_validation="when_required",
                    ),
                )


A few key points about what that code does:
1.  Access to any data on AWS requires "credentials".   Be grateful Earthscope simplified that for you behind that "get_aws_credentials" call.
2.  AWS uses the term "bucket" to describe a collection of stuff.   You should think of a bucket as being like top level directory for disk storage in a unix file system.   That sets the symbol "BUCKET" to a thing we can use to define that abstract entry point.
3.  A "Session" is more or less the equivalent of a login shell on a unix computer.   It holds data boto3 uses to tell the system you are a valid user and what that user is allowed to do.
4.  The "s3_client" symbol holds yet another client application.   Like the database client all interactions with s3 must occur through that agent.

### Very important warning
MAY NO LONGER BE TRUE - QUESTION PENDIBG.

Currently the s3 client we are using has a 30 minute timeout.   If something in this notebook mysterious throws an error, return code box 1 above and try the code box again.   The only code box that intentionally throws an error in this notebook is box 7 and the text tells you it is expected to throw an error.   The timeout problem is highly likely as I am planning to spend more than 30 minutes working through it.   I will probably get the error before you do.   Anyone running this exercise outside the class period will have different timing. 

## Earthscope S3 archive structure
The boto3 tool for examining the content of a give "bucket" is called "list_objects_v2".  It is loosely equivalent to the unix "ls" command or the windows "dir" command.   That is, it's purpose is to view the content of things (objects) that have some prefined relationship.   Earthscope designed their store to mimic a unix file system.  That is done on S3 by using what they call a "Prefix".  

The top level prefix for seismology waveform data at Earthscope is "miniseed".   Think of the prefix "miniseed" as like a file system with the name "miniseed".   i.e. the following is the boto3 equivalent of running `ls /miniseed` on a computer with a file system mounted as "/miniseed":

In [2]:
list_resp = s3_client.list_objects_v2(Bucket=BUCKET, Prefix="miniseed/", Delimiter="/")

Now `list_resp` contains a lot of stuff.   It is packaged into a dictionary as seen in the next code box:

In [3]:
print("The type of list_resp is ",type(list_resp))
print("The keys of the dictionary are")
for k in list_resp.keys():
    print(f"   {k}")

The type of list_resp is  <class 'dict'>
The keys of the dictionary are
   ResponseMetadata
   IsTruncated
   Contents
   Name
   Prefix
   Delimiter
   MaxKeys
   CommonPrefixes
   EncodingType
   KeyCount


All but two of these are things only AWS cares about.   The equivalent of a directory is the content of the "CommonPrefixes".   The entire thing for this store is pretty huge, but consider this:

In [4]:
print("The type of the content of CommonPrefixes is",type(list_resp["CommonPrefixes"]))
print("The first component of that list is ",list_resp["CommonPrefixes"][0])

The type of the content of CommonPrefixes is <class 'list'>
The first component of that list is  {'Prefix': 'miniseed/12/'}


In words, what that shows is CommonPrefixes references a list of dictionaries with only one item.   Those who know too well about SEED station codes may guess that "12" is a net code.  We can take that one step further using this incantation from the Earthscope SDK documenation:

In [5]:
nets = [c["Prefix"].split("/", 1)[1] for c in list_resp["CommonPrefixes"]]
print(nets)

['12/', '14/', '16/', '17/', '1A/', '1B/', '1C/', '1D/', '1E/', '1F/', '1G/', '1H/', '1J/', '1K/', '1L/', '1M/', '1P/', '1Q/', '1R/', '1T/', '1U/', '1V/', '1W/', '1Z/', '22/', '24/', '28/', '29/', '2A/', '2B/', '2C/', '2D/', '2E/', '2F/', '2G/', '2H/', '2I/', '2J/', '2K/', '2L/', '2M/', '2O/', '2P/', '2Q/', '2T/', '2U/', '2V/', '34/', '3A/', '3B/', '3C/', '3D/', '3E/', '3F/', '3H/', '3J/', '3K/', '3L/', '3R/', '3U/', '3W/', '3Y/', '4A/', '4B/', '4E/', '4F/', '4H/', '4I/', '4J/', '4K/', '4N/', '4O/', '4P/', '4Q/', '4R/', '4S/', '4T/', '4U/', '4Y/', '4Z/', '5A/', '5B/', '5C/', '5E/', '5F/', '5G/', '5H/', '5I/', '5J/', '5K/', '5L/', '5O/', '5P/', '5Q/', '5S/', '5W/', '5X/', '6A/', '6B/', '6C/', '6D/', '6E/', '6F/', '6G/', '6H/', '6I/', '6J/', '6K/', '6L/', '6M/', '6N/', '6O/', '6P/', '6Q/', '6R/', '6W/', '7A/', '7B/', '7C/', '7D/', '7E/', '7F/', '7G/', '7I/', '7J/', '7K/', '7L/', '7O/', '7P/', '7Q/', '7S/', '7T/', '7U/', '8A/', '8B/', '8E/', '8F/', '8G/', '8H/', '8I/', '8J/', '8L/', '8M/'

The key thing I want you to recognize here is that CommonPrefixes is a grouping construct that Earthscope uses to mimic the file structure of the archive they previously stored on servers in Seattle.   Note, by the way that wasn't an actual file structure either.   Since the earliest days of archival storage IRIS used a system that mimiced a unix file system with the actual data stored on tape and later a large disk array (I think).   

We could drill down through the hierarchival grouping, but I am hoping you will follow me if I leap to say the waveform data are organized locally as day volumes for a given fdsn station.   The "prefix" logical structure is:
```
   miniseed/net/year/day
```
where `net` is a SEED net code, `year` is what the english word means, and `day` is the julian day of the year.  

The above construct was the equivalent of running the unix "ls" command on a directory BUT CommonPrefixes returns only links to subgroups (the equivalent of a subdirectory).  Leaf objects, which are the equivalent of files, are accessed with key "Contents" instead of "CommonPrefixes".   For example, here is the content for Iris Ida (net code II) for day 255 of 2000.

In [6]:
# note the trailing / is essential for this to work
list_resp = s3_client.list_objects_v2(Bucket=BUCKET, Prefix="miniseed/II/2000/255/", Delimiter="/")
keys=[]
for obj in list_resp['Contents']:
    keys.append(obj['Key'])
for k in keys:
    print(k)

miniseed/II/2000/255/AAK.II.2000.255#1
miniseed/II/2000/255/ALE.II.2000.255#1
miniseed/II/2000/255/ARU.II.2000.255#1
miniseed/II/2000/255/ASCN.II.2000.255#1
miniseed/II/2000/255/BFO.II.2000.255#1
miniseed/II/2000/255/BORG.II.2000.255#1
miniseed/II/2000/255/BRVK.II.2000.255#1
miniseed/II/2000/255/COCO.II.2000.255#1
miniseed/II/2000/255/EFI.II.2000.255#1
miniseed/II/2000/255/FFC.II.2000.255#1
miniseed/II/2000/255/HOPE.II.2000.255#1
miniseed/II/2000/255/KAPI.II.2000.255#1
miniseed/II/2000/255/KDAK.II.2000.255#1
miniseed/II/2000/255/KIV.II.2000.255#1
miniseed/II/2000/255/KURK.II.2000.255#1
miniseed/II/2000/255/KWAJ.II.2000.255#1
miniseed/II/2000/255/LVZ.II.2000.255#1
miniseed/II/2000/255/MBAR.II.2000.255#1
miniseed/II/2000/255/MSEY.II.2000.255#1
miniseed/II/2000/255/NIL.II.2000.255#1
miniseed/II/2000/255/NNA.II.2000.255#1
miniseed/II/2000/255/OBN.II.2000.255#1
miniseed/II/2000/255/PFO.II.2000.255#1
miniseed/II/2000/255/RAYN.II.2000.255#1
miniseed/II/2000/255/SACV.II.2000.255#1
miniseed/II/

Note in the Earthscope archives prefixes like our example (Prefix="miniseed/II/2000/255/") have no equivalents of subdirectories.   If you try to look at "CommonPrefixes" in the output of list_objects_v2 you will get an error as show here:

In [7]:
# note the trailing / is essential for this to work
list_resp = s3_client.list_objects_v2(Bucket=BUCKET, Prefix="miniseed/II/2000/255/", Delimiter="/")
print(list_resp["CommonPrefixes"])

KeyError: 'CommonPrefixes'

The more important point is that a name like "miniseed/II/2000/255/PFO.II.2000.255#1" should be viewed as a unique string to identify a particular set of waveform data.   In S3 that name is the key to fetch what you want.   The fact that looks like a unix path name is an artifact of how Earthscope chose to organize the archive.   You can translate "miniseed/II/2000/255/PFO.II.2000.255#1" into this description:   That string is an s3 object name pointing to continuous, miniseed data for IRIS Ida (II) station "PFO" recorded on day 255 of 2000.  

The Earthscope archive has a large number of object names of that general form.   Look at that list of network code and realize the archive spans nearly 40 years and there is one item for every station-day.   

## Retrieving waveform data
As we just saw the waveform data are packaged into miniseed bundles organized by station-day.   Let's next pull one of those and look into the content.   

The tool for retrieving the raw bits for an s3 object is called, what else, `get_object`.   Here is a direct application to that same PFO data we referenced above in the simplest form possible.

In [8]:
s3obj = s3_client.get_object(Bucket=BUCKET,
                                Key="miniseed/II/2000/255/PFO.II.2000.255#1",
                            )
data = s3obj["Body"].read()

Noting, either of those lines could fail with an exception being thrown. 

Let's look at a couple features of what that gave us.

In [9]:
print("Type of what the read method returned=",type(data))
print("The size of the content is ",len(data)," bytes")

Type of what the read method returned= <class 'bytes'>
The size of the content is  20389888  bytes


So, that thing isn't tiny.   If you are python expert you know that a "bytes" means a raw block of binary data.   In this case, that data is a miniseed format sequence of packets.   The following incantation uses obspy's miniseed reader to crack that block of data and turn it into an obspy Stream object.

In [10]:
import obspy
import io

strm = obspy.read(io.BytesIO(data),format="mseed")
print(strm)

16 Trace(s) in Stream:
II.PFO.00.BHE | 2000-09-11T00:00:00.006900Z - 2000-09-11T23:59:59.955777Z | 20.0 Hz, 1727998 samples
II.PFO.00.BHN | 2000-09-11T00:00:00.006900Z - 2000-09-11T23:59:59.955777Z | 20.0 Hz, 1727998 samples
II.PFO.00.BHZ | 2000-09-11T00:00:00.006900Z - 2000-09-11T23:59:59.955777Z | 20.0 Hz, 1727998 samples
II.PFO.00.LHE | 2000-09-11T00:00:00.006800Z - 2000-09-11T23:59:59.099496Z | 1.0 Hz, 86400 samples
II.PFO.00.LHN | 2000-09-11T00:00:00.006800Z - 2000-09-11T23:59:59.099496Z | 1.0 Hz, 86400 samples
II.PFO.00.LHZ | 2000-09-11T00:00:00.006800Z - 2000-09-11T23:59:59.099496Z | 1.0 Hz, 86400 samples
II.PFO.00.LNE | 2000-09-11T00:00:00.416900Z - 2000-09-11T23:59:59.519896Z | 1.0 Hz, 86400 samples
II.PFO.00.LNN | 2000-09-11T00:00:00.416900Z - 2000-09-11T23:59:59.519896Z | 1.0 Hz, 86400 samples
II.PFO.00.LNZ | 2000-09-11T00:00:00.416900Z - 2000-09-11T23:59:59.519896Z | 1.0 Hz, 86400 samples
II.PFO.00.VHE | 2000-09-11T00:00:00.006700Z - 2000-09-11T23:59:50.095525Z | 10.0 s, 86

PFO is a GSN station so there are a ton of channels:  two sensors (loc codes 00 and 10) with B channels, L and V channels, and some state of health channels.   Also note from the time ranges this confirms the data are day long segments.   

The code you ran before the start of this class did a more robust version of that read sequence thousands of times.   To mesh with MsPASS the code you ran did something similar to the following:

In [11]:
from mspasspy.util.converter import Stream2TimeSeriesEnsemble
strm = strm.select(channel="B*")
ens = Stream2TimeSeriesEnsemble(strm)
print("TimeSeriesEnsemble content")
print("net sta chan loc starttime")
for d in ens.member:
    print(d['net'],d['sta'],d['chan'],d['loc'],obspy.UTCDateTime(d.t0))

TimeSeriesEnsemble content
net sta chan loc starttime
II PFO BHE 00 2000-09-11T00:00:00.006900Z
II PFO BHN 00 2000-09-11T00:00:00.006900Z
II PFO BHZ 00 2000-09-11T00:00:00.006900Z
II PFO BH1 10 2000-09-11T00:00:00.001900Z
II PFO BH2 10 2000-09-11T00:00:00.001900Z
II PFO BHZ 10 2000-09-11T00:00:00.001900Z


## Parallel Access with dask
Remember in session 2 that we used the MongoDBWorker object to make the database service work cleanly on workers.   We have a similar problem here on GeoLab is we want to use dask to access s3.   Dask workers are dumb and know nothing about s3 unless we tell them how to access s3.   You might think you could just write a simple parallel reader like the following that could be used with a map operator:

```
def s3_parallel_reader(s3_client,s3object):
    """
    Example of how NOT to create a parallel reader using s3 client passed as arg0 (s3_client) and the object
    key string passed as arg1 (s3object).
    """
```

If you try that you will find it fails because s3_client cannot be "serialized".   

You might also think you could just create an s3_client inside the reader like this:
```
def s3_parallel_reader(s3object):
    client = EarthScopeClient()
    creds = client.user.get_aws_credentials()

    S3_ACCESS_POINT = "earthscope-mseed-res-na3mtd4fq5kz7pntcyr1uh46use2a--ol-s3"
    BUCKET = S3_ACCESS_POINT

    session = boto3.Session(
        aws_access_key_id=creds.aws_access_key_id,
        aws_secret_access_key=creds.aws_secret_access_key.get_secret_value(),
        aws_session_token=creds.aws_session_token.get_secret_value(),
    )
    s3_client = session.client(
                    "s3",
                    config=Config(
                        request_checksum_calculation="when_required",
                        response_checksum_validation="when_required",
                    ),
                )
    s3obj = s3_client.get_object(Bucket=BUCKET,
                                Key=s3object,
                            )
    data = s3obj["Body"].read()
    return data
```
That might actually run, but you would find out two things very quickly:
1.  It would run very very slowly because every read requires instantiating an s3_client, which is not trivial.
2.  The virtual machine you are running would probably complain about the high traffic you would generate this way.   Every time you instantiate a client a rather large volume of computer bodily fluids are exchanged between the S3 service and the client you are creating.  That exchange, in fact, is why instantiating an s3 client is very time consuming.  

The code you ran before the class solves this issue with a worker plugin that has a structure similar to MongoDBWorker.   You could import my prototype code from the local module "s3_worker_plugin", but I think it would be instructive for you to see it.  So, we'll create a local copy in this notebook in the next box and then we'll see how to use it. 

In [12]:
from dask.distributed import WorkerPlugin, get_worker
class S3Worker(WorkerPlugin):
    def __init__(self,key="s3client"):
        self.worker_key=key
    def setup(self,worker):
        client = EarthScopeClient()
        creds = client.user.get_aws_credentials()
        session = boto3.Session(
            aws_access_key_id=creds.aws_access_key_id,
            aws_secret_access_key=creds.aws_secret_access_key.get_secret_value(),
            aws_session_token=creds.aws_session_token.get_secret_value(),
        )

        s3_client = session.client(
                    "s3",
                    config=Config(
                        request_checksum_calculation="when_required",
                        response_checksum_validation="when_required",
                    ),
)

        worker.data[self.worker_key] = s3_client
    def teardown(self,worker):
        s3_client = worker.data.get(self.worker_key)
        s3_client.close()

We don't currently have a dask client accessible, so we need the stock MsPASS client creation you've used in the previous sessions. 

In [13]:
from mspasspy.db.database import Database
import mspasspy.client as msc
mspass_client = msc.Client()
dask_client = mspass_client.get_scheduler()

Noting in this case we aren't using MongoDB yet so we don't need to create a database handle.  

We enable the plugin the same way we did the MongoDBWorker plugin in session 2 but use an insance of S3Worker instead of a MongoDBWorker.

In [14]:
s3worker = S3Worker()
# this may not be necessary, but this assures the workers have the code the need to fully create the plugin
dask_client.upload_file("s3_worker_plugin.py")
dask_client.register_plugin(s3worker)

{'tcp://127.0.0.1:36495': {'status': 'OK'},
 'tcp://127.0.0.1:36883': {'status': 'OK'},
 'tcp://127.0.0.1:42015': {'status': 'OK'},
 'tcp://127.0.0.1:45429': {'status': 'OK'}}

Look back at your Session2 notebook and you will see register_plugin returned a similar output in the same construct.  That dictionary is just the url of the network address of each dask worker.

I'll show you how this works in two ways:
1.  Below is a do nothing script that reads and reports content of all II stations for the random day 255 of 2000 we probed above.
2.  We'll close by digging into the code you ran before the start of the class.  It illustrates more robust approaches than the crude but simple code I'll show you below.

First, our reader will need this other function extracted from the local s3_worker_plugin module.  The docstring says what it does.  You'll see how we use it in the box that follows.

In [17]:
def fetch_s3_client(session=None,worker_data_key="s3client"):
    """
    Generic tool to fetch s3 client for Earthscope s3 archive.  
    
    When session is None (default) the function assumes it is running in a parallel environment with dask. 
    In that situation it assumes workers have been previously initialized with the WorkerPlugin "S3Worker"
    and it can fetch the client from the worker data area with the key defined by "worker_data_key".
    
    When session is defined it is used to instantiate a client using the stock Earthscope incantation 
    to fetch the access point. 
    """
    if session is None:
        try:
            worker = get_worker()
        except Exception as e:
            raise ValueError(
                    "fetch_s3_client: this function must be "
                    "executed within a Dask worker context so that get_worker() succeeds and an "
                    "s3 client is available via a worker plugin."
                ) from e
        try:
            s3_client = worker.data[worker_data_key]

        except KeyError as e:
            message = "fetch_s3_client:  dask worker does not have a valued defined for expected key={}\n".format(worker_data_key)
            message = "Make sure you run S3Worker before you submit anything to the dask cluster"
            raise ValueError(message) from e
    else:
        s3_client = session.client(
                    "s3",
                    config=Config(
                        request_checksum_calculation="when_required",
                        response_checksum_validation="when_required",
                    ),
                )


    return s3_client

These three functions define our do nothing processing sequence.   We'll use them in both the serial and parallel codes below.   

A detail to note here.  Often with mspass workflows for simple problems using just print statements can help identify where a problem is happening.  I'm leaving in the scaffolding I used do debug these three functions and the serial script I used to test it.  All the lines with "#print" are my debug lines.  I removed them when I got this to work.

In [15]:
from mspasspy.ccore.seismic import TimeSeriesEnsemble
from mspasspy.ccore.utility import ErrorSeverity
from mspasspy.util.seismic import number_live

def s3_parallel_reader(s3objectname,session=None):
    """
    Function with the same signature as the bad example given above, but which uses the worker plugin to 
    load the s3 client cleanly. 
    """
    s3_client = fetch_s3_client(session)
    # this time we do need a basic error handler 
    try:
        s3obj = s3_client.get_object(Bucket=BUCKET,
                                Key=s3objectname
                            )
        data = s3obj["Body"].read()
        return data
    except Exception as e:
        # generic handler 
        print("s3_parallel_reader failed with requested object name=",s3objectname)
        return None

def convert_raw_data(bytedata)->TimeSeriesEnsemble:
    """
    Converts raw miniseed data to a MsPASS TimeSeriesEnsemble.   Handles failures cleanly.
    Not generic as it returns only B channels.   Does show the general pattern of 
    how to make a parallel function bombproof with mspass error logging.
    """
    # this is a variant of what we did above to crack the byte stream.  
    # the main difference is handling None inputs
    if bytedata is None:
        out = TimeSeriesEnsemble()
        out.elog.log_error("convert_raw_data","reader failed",ErrorSeverity.Invalid)
        # not technically necessary but clearer to do this
        out.kill()
    else:
        try:
            strm = obspy.read(io.BytesIO(bytedata),format="mseed")
            #print(strm)
            strm = strm.select(channel="B*")    # hard code selecting only B channels
            #print(strm)
            ens = Stream2TimeSeriesEnsemble(strm)
            #print("size from converter=",len(ens.member))
            del strm   # good practice for memory management in any parallel function
            return ens
        except Exception as e:
            message = "conversion failed.  Exception message = {}".format(str(e))
            out = TimeSeriesEnsemble()
            out.elog.log_error("convert_raw_data",message,ErrorSeverity.Invalid)
            out.kill()
            return out
        
def summarize(ens):
    """
    Handles return of convert_raw_data returning a message about data read for each station. 
    We do or the volume of data returned by an entire network day would be pretty large.
    """
    if ens.live:
        nmem = len(ens.member)
        sta = ens.member[0]['sta']
        nlive = number_live(ens)
        outstr = f"{sta} {nmem} {nlive}"
    else:
        outstr="reader failure on this datum.  elog content={}".format(str(ens.elog.get_error_log()))
    return outstr
        

First, like always it is useful to test a parallel code with serial prototype.   It will be useful here anyway to evaluate the speedup with 4 workers compared to a serial version.

We rebuild the list of object names and loop over that to do this do nothing processing.

In [18]:
import time
# copied from above 
list_resp = s3_client.list_objects_v2(Bucket=BUCKET, Prefix="miniseed/II/2000/255/", Delimiter="/")
keys=[]
for obj in list_resp['Contents']:
    keys.append(obj['Key'])
print("station number_loaded number_live")
t0 = time.time()
for k in keys:
    #print(k)
    # note we need to define session for the serial job to make the fetch_s3_client work correctly
    draw = s3_parallel_reader(k,session)
    #print(len(draw))
    ens = convert_raw_data(draw)
    #print(len(ens.member))
    out = summarize(ens)
    print(out)
t=time.time()
print("Time to run serial job=",t-t0)
    

station number_loaded number_live
AAK 3 3
ALE 3 3
ARU 3 3
ASCN 3 3
BFO 3 3
BORG 6 6
BRVK 3 3
COCO 9 9
EFI 3 3
FFC 3 3
HOPE 10 10
KAPI 6 6
KDAK 18 18
KIV 3 3
KURK 6 6
KWAJ 6 6
LVZ 6 6
MBAR 6 6
MSEY 8 8
NIL 5 5
NNA 3 3
OBN 6 6
PFO 6 6
RAYN 6 6
SACV 5 5
SHEL 6 6
SUR 8 8
TAU 3 3
TLY 3 3
WRAB 11 11
Time to run serial job= 15.709824562072754


Now the parallel version.  The one complication is removing session from s3_parallel_reader.

In [19]:
import dask.bag as dbg
t0=time.time()
mybag = dbg.from_sequence(keys)
mybag = mybag.map(s3_parallel_reader)
mybag = mybag.map(convert_raw_data)
mybag = mybag.map(summarize)
out = mybag.compute()
t = time.time()
print("Execution time=",t-t0)
print("station number_loaded number_live")
for line in out:
    print(line)

Execution time= 7.566828489303589
station number_loaded number_live
AAK 3 3
ALE 3 3
ARU 3 3
ASCN 3 3
BFO 3 3
BORG 6 6
BRVK 3 3
COCO 9 9
EFI 3 3
FFC 3 3
HOPE 10 10
KAPI 6 6
KDAK 18 18
KIV 3 3
KURK 6 6
KWAJ 6 6
LVZ 6 6
MBAR 6 6
MSEY 8 8
NIL 5 5
NNA 3 3
OBN 6 6
PFO 6 6
RAYN 6 6
SACV 5 5
SHEL 6 6
SUR 8 8
TAU 3 3
TLY 3 3
WRAB 11 11


## Study Precourse Code
The last thing I want us to do today is study the code you ran before the class began to assemble the data set we've been working on this week.  

Open the python file "s3segment_reader.py" and I'll lead you through what it does.  It does something much more elaborate than the simple tutorial code you just ran.